# Resultados, límites y linaje · 2.1.4

compile construye; run/arun ejecutan. RunResult conserva contrato común, IDs y evidencia; lineage ofrece vistas humanas y para agentes.

In [2]:
import pydantic
import agentic_systems as toolkit
assert toolkit.__version__ == '2.1.4'


In [3]:
class State(pydantic.BaseModel):
    value: int

@toolkit.tool
def increment(value: int) -> dict:
    return {'value': value + 1}
graph = toolkit.Graph(State)
graph.add_node('increment', increment)
graph.add_edge(toolkit.START, 'increment')
graph.add_edge('increment', toolkit.END)
agent = toolkit.Agent('counter', graph=graph)


In [4]:
import asyncio, json
program = agent.compile()
result = program.run({'value': 3}, timeout_s=10)
async_result = await program.arun({'value': 3})
restored = toolkit.RunResult(**json.loads(json.dumps(result.to_dict())))
assert result.ok and async_result.data == result.data and restored.check_invariants().ok
print(restored.human_results())
print(restored.lineage().to_prompt_context())


counter: COMPLETED
Execution: run-6fa9e77f478748cfb1a1029f3bf0b0f1
Input: {"value":3}
Output: {"value":4}
Process:
- Tool increment: completed -> {"value":4}
{"ok":true,"steps":[{"step_id":"run-6fa9e77f478748cfb1a1029f3bf0b0f1","kind":"execution","title":"counter","summary":"completed","source":"python-runtime","why":"Recorded execution evidence","evidence":{"ok":true,"data":{"value":4},"errors":[]}},{"step_id":"run-7621d30d89284c46b598c1553422052a","kind":"execution","title":"increment","summary":"completed","source":"python-runtime","why":"Recorded execution evidence","evidence":{"ok":true,"data":{"value":4},"errors":[]}},{"step_id":"tool-e06426f40ad046bba86aafd432f6c11b","kind":"tool","title":"increment","summary":"completed","source":"increment","why":"Recorded Tool event","evidence":{"ok":true,"output":{"data":{"value":4}},"error":null}}],"omitted_steps":0,"failed_steps":0,"execution_id":"run-6fa9e77f478748cfb1a1029f3bf0b0f1","answer":"{\"value\": 4}"}
